# RAG Embedding Model Analysis

Thesis experiments — loading results from `data/runs/`.

## Setup

In [ ]:
import sys, json, csv, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "figure.figsize": (10, 5),
})

RESULTS_DIR = Path.cwd().parent / "data" / "runs"
print(f"Results directory: {RESULTS_DIR}")
print(f"Runs found: {len(list(RESULTS_DIR.glob('*.csv')))}")


## 1. Load Results

Load all CSV files from `data/runs/` and display available runs.

In [ ]:
csv_files = sorted(RESULTS_DIR.glob("*.csv"))
agg_csv_files = sorted(RESULTS_DIR.glob("*_aggregate.csv"))
run_csv_files = [f for f in csv_files if "_aggregate" not in f.name and "_meta" not in f.name]

print(f"Full result CSVs ({len(run_csv_files)}):")
for f in run_csv_files:
    meta = RESULTS_DIR / f.name.replace(".csv", "_meta.json")
    meta_info = ""
    if meta.exists():
        m = json.loads(meta.read_text())
        meta_info = f"  — {m.get('run_name', '')}  top_k={m.get('top_k', '?')}  ({m.get('num_queries', '?')} queries)"
    print(f"  {f.name}{meta_info}")

# Load the most recent run
if run_csv_files:
    latest = run_csv_files[-1]
    print(f"\nLoading latest: {latest.name}")
    df = pd.read_csv(latest)
    print(f"  {len(df)} rows x {len(df.columns)} columns")
    display(df.head())
else:
    df = pd.DataFrame()
    print("No result files found — run an experiment first")


## 2. Aggregate Statistics

Per-model mean ± std for each metric across all queries.

In [ ]:
if not df.empty:
    metric_cols = ["hit_rate","mrr","precision","ndcg","exact_match",
                   "rouge_l_f1","semantic_similarity","faithfulness",
                   "answer_relevancy","llm_quality_score"]
    
    available = [c for c in metric_cols if c in df.columns]
    
    def safe_float(x):
        try: return float(x)
        except: return np.nan
    
    for c in available:
        df[c] = df[c].apply(safe_float)
    
    agg = df.groupby("model")[available].agg(["mean","std","count"])
    # Round
    for metric in available:
        if (metric, "mean") in agg.columns:
            agg[(metric, "mean")] = agg[(metric, "mean")].round(4)
        if (metric, "std") in agg.columns:
            agg[(metric, "std")] = agg[(metric, "std")].round(4)
    
    display(agg)
    
    # Also show the best model per metric
    best = {}
    for metric in available:
        means = df.groupby("model")[metric].mean()
        if not means.empty:
            best[metric] = means.idxmax()
    print("\nBest model per metric:")
    for k, v in best.items():
        print(f"  {k:25s} → {v}")
else:
    print("No data loaded.")


## 3. Model Comparison Bar Chart

Compare all models across key metrics.

In [ ]:
if not df.empty:
    key_metrics = [c for c in ["hit_rate","mrr","semantic_similarity","faithfulness"] if c in available]
    if key_metrics:
        fig, axes = plt.subplots(1, len(key_metrics), figsize=(14, 4))
        if len(key_metrics) == 1:
            axes = [axes]
        
        for ax, metric in zip(axes, key_metrics):
            means = df.groupby("model")[metric].mean().sort_values()
            colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(means)))
            ax.barh(range(len(means)), means.values, color=colors)
            ax.set_yticks(range(len(means)))
            ax.set_yticklabels(means.index, fontsize=8)
            ax.set_xlabel(metric.replace("_", " ").title())
            ax.set_xlim(0, 1.05)
            ax.grid(axis="x", alpha=0.3)
        
        plt.suptitle("Model Comparison Across Metrics", fontsize=14)
        plt.tight_layout()
        plt.show()


## 4. Per-Dataset Performance

If multiple datasets are present, show dataset-level breakdown.

In [ ]:
if not df.empty and "dataset" in df.columns:
    datasets = df["dataset"].unique()
    if len(datasets) > 1:
        fig, ax = plt.subplots(figsize=(12, max(4, len(datasets) * 0.4)))
        
        metric = "semantic_similarity" if "semantic_similarity" in available else available[0]
        
        pivot = df.pivot_table(index="dataset", columns="model", values=metric, aggfunc="mean")
        pivot = pivot.round(4)
        display(pivot)
        
        im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, fontsize=8, rotation=45, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=8)
        ax.set_title(f"{metric.replace('_', ' ').title()} by Dataset and Model")
        plt.colorbar(im, ax=ax, fraction=0.05)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Single dataset: {datasets[0]}")
else:
    print("No dataset column in data.")


## 5. Top-K Comparison (if multiple top-K values)

If the run name encodes top-K information, group and compare.

In [ ]:
def extract_topk(filename: str) -> int | None:
    m = re.search(r'_k([0-9]+)', filename)
    return int(m.group(1)) if m else None

if run_csv_files:
    topk_groups = {}
    for f in run_csv_files:
        k = extract_topk(f.name)
        if k is not None:
            topk_groups.setdefault(k, []).append(f)
    
    if len(topk_groups) > 1:
        fig, ax = plt.subplots(figsize=(8, 4))
        markers = {"hit_rate": "o", "mrr": "s", "semantic_similarity": "^"}
        
        for metric, marker in markers.items():
            if metric not in available:
                continue
            means = []
            for k in sorted(topk_groups.keys()):
                # Load first CSV for each top-k
                csv_path = topk_groups[k][0]
                d = pd.read_csv(csv_path)
                for c in [metric]:
                    d[c] = d[c].apply(safe_float)
                means.append(d[metric].mean())
            ax.plot(sorted(topk_groups.keys()), means, marker=marker, label=metric)
        
        ax.set_xlabel("Top-K")
        ax.set_ylabel("Mean Score")
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_title("Metric Scores vs Top-K")
        plt.show()


## 6. LaTeX Table Export

Generate LaTeX tables ready for thesis inclusion.

In [ ]:
def latex_table(df_agg, caption="Aggregate results", label="tab:results"):
    """Generate a LaTeX table string from aggregated results."""
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\begin{tabular}{l" + "c" * (len(df_agg.columns)//2) + "}",
        r"\toprule",
        " & ".join(df_agg.columns.get_level_values(0)) + r" \\",
        r"\midrule",
    ]
    for idx in df_agg.index:
        row_vals = []
        for col in df_agg.columns:
            v = df_agg.loc[idx, col]
            if isinstance(v, float):
                row_vals.append(f"{v:.3f}")
            else:
                row_vals.append(str(v))
        lines.append(" & ".join([str(idx)] + row_vals) + r" \\")
    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)

if not df.empty:
    # Build multi-level LaTeX table
    # Group by model, show mean ± std
    latex_rows = []
    for model in sorted(df["model"].unique()):
        row = {"Model": model}
        for metric in available:
            vals = df[df["model"] == model][metric].dropna()
            if not vals.empty:
                row[f"{metric}"] = f"{vals.mean():.3f} ± {vals.std():.3f}"
            else:
                row[f"{metric}"] = "—"
        latex_rows.append(row)
    
    latex_df = pd.DataFrame(latex_rows)
    print(latex_df.to_string(index=False))
    
    # Generate LaTeX
    latex = latex_table(
        agg,
        caption="Per-model performance across all metrics",
        label="tab:model_comparison",
    )
    print("\n\n--- LaTeX table ---\n")
    print(latex)


## 7. Per-Metric Best Model Summary

In [ ]:
if not df.empty:
    summary = []
    for metric in available:
        means = df.groupby("model")[metric].mean().dropna()
        if means.empty:
            continue
        best_model = means.idxmax()
        best_score = means.max()
        worst_model = means.idxmin()
        worst_score = means.min()
        summary.append({
            "Metric": metric.replace("_", " ").title(),
            "Best Model": best_model,
            f"Best Score": f"{best_score:.3f}",
            "Worst Model": worst_model,
            f"Worst Score": f"{worst_score:.3f}",
            "Spread": f"{best_score - worst_score:.3f}",
        })
    
    summary_df = pd.DataFrame(summary)
    display(summary_df)
    print("\n--- LaTeX ---\n")
    print(summary_df.to_latex(index=False, escape=False))


## 8. Save Analysis

Export figures as PNG for thesis inclusion.

In [ ]:
# Save exports to a figures directory
fig_dir = RESULTS_DIR / "figures"
fig_dir.mkdir(exist_ok=True)

# Re-generate and save each figure
if not df.empty and key_metrics:
    fig, axes = plt.subplots(1, len(key_metrics), figsize=(14, 4))
    if len(key_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, key_metrics):
        means = df.groupby("model")[metric].mean().sort_values()
        colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(means)))
        ax.barh(range(len(means)), means.values, color=colors)
        ax.set_yticks(range(len(means)))
        ax.set_yticklabels(means.index, fontsize=8)
        ax.set_xlabel(metric.replace("_", " ").title())
        ax.set_xlim(0, 1.05)
        ax.grid(axis="x", alpha=0.3)
    plt.suptitle("Model Comparison Across Metrics", fontsize=14)
    plt.tight_layout()
    fig.savefig(fig_dir / "model_comparison.png", dpi=200, bbox_inches="tight")
    print(f"Saved: {fig_dir / 'model_comparison.png'}")

print("\nDone. Figures saved to", fig_dir)
